# Smoke test: value-guided KV cache engine

First end-to-end run: load DeepSeek-R1-Distill-Qwen-7B, run a handful of GSM8K
examples through the custom decode loop under each baseline policy plus the
entropy-salience policy, and sanity-check accuracy/throughput/memory numbers
before scaling up to a full eval sweep.

Run this on the remote GPU box (L4/L40). Install deps first:
`pip install -r requirements.txt && pip install -e .`

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
from vgkv.model import load_model_and_tokenizer, DEFAULT_MODEL_ID
from vgkv.eval.gsm8k import load_gsm8k, build_prompt, is_correct
from vgkv.decode import generate_with_policy, DecodeConfig
from vgkv.value_models import NoEviction, RandomPolicy, RecencyPolicy, H2OPolicy, EntropySaliencePolicy

print(torch.cuda.get_device_name(0))

In [ ]:
model, tokenizer = load_model_and_tokenizer(DEFAULT_MODEL_ID, device="cuda")

In [ ]:
ds = load_gsm8k(split="test", limit=5)
example = ds[0]
prompt = build_prompt(example)
print(prompt)

## Single-example sanity check across policies

Use a generous budget first (>= expected total sequence length) with `NoEviction`
to confirm the decode loop + eager attention plumbing works and matches
`model.generate()` output for the same prompt (greedy decoding should be
identical). Then drop the budget and compare policies.

In [ ]:
config_full = DecodeConfig(max_new_tokens=256, budget=10_000, sink_size=4)
text, metrics = generate_with_policy(model, tokenizer, prompt, NoEviction(), config_full)
print(text)
print(metrics.as_dict())
print("correct:", is_correct(text, example))

In [ ]:
# cross-check against generate() for identical greedy output on the same prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    ref_out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
ref_text = tokenizer.decode(ref_out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(ref_text)
print("matches custom loop:", ref_text == text)

## Tight-budget comparison across policies

Pick a budget well below the full sequence length so eviction actually kicks in,
then compare accuracy/throughput/memory across policies on a few examples.

In [ ]:
policies = {
    "no_eviction": NoEviction(),
    "random": RandomPolicy(seed=0),
    "recency_lru": RecencyPolicy(),
    "h2o": H2OPolicy(),
    "entropy_salience": EntropySaliencePolicy(),
}

tight_budget = 128
results = []
for name, policy in policies.items():
    cfg = DecodeConfig(max_new_tokens=256, budget=tight_budget, sink_size=4)
    text, metrics = generate_with_policy(model, tokenizer, prompt, policy, cfg)
    correct = is_correct(text, example)
    row = {"policy": name, "correct": correct, **metrics.as_dict()}
    results.append(row)
    print(name, "correct:", correct, "tok/s:", round(metrics.decode_tokens_per_sec, 1))

import pandas as pd
pd.DataFrame(results)

## Next steps

- Scale from 1 example to a subset (e.g. 50-100) of GSM8K test, average accuracy
  and throughput per policy
- Sweep budget (e.g. 64/128/256/512/1024) to build the accuracy-vs-budget curve
  per policy
- Log results to `../results/` as CSV/JSON for plotting
- Update `../PROJECT.md` section 6 approach log with findings before trying the
  next value-model idea (attention-graph centrality, PRM-based step scoring)